In [87]:
# Agentic workflow: UI ---> deal agent framework ---> autonomous planning agent ---> scanner agent ---> ensemble agent ---> messaging agent
import os
import sys
import re
import logging
import json
import time
from openai import OpenAI
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion
import litellm
import modal
import requests
from typing import Optional, List, Dict, Self
from concurrent.futures import ThreadPoolExecutor, as_completed
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate
import logging
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import HashingVectorizer
from pydantic import BaseModel, Field
from bs4 import BeautifulSoup
import feedparser
from tqdm import tqdm
import requests
import gradio as gr
import queue
import threading

In [2]:
# Important to skip SSL certificate verification 
import truststore
truststore.inject_into_ssl() 

In [3]:
root = logging.getLogger()

In [4]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check the keys
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

OpenAI API Key exists and begins sk-proj-


In [5]:
# Connect to OpenAI client library
openai = OpenAI()

In [6]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
# Let's build ensemble agent

In [8]:
# Utility code for logging colors
# This is utiltiy for logging colors
# Foreground colors
RED = '\033[31m'
GREEN = '\033[32m'
YELLOW = '\033[33m'
BLUE = '\033[34m'
MAGENTA = '\033[35m'
CYAN = '\033[36m'
WHITE = '\033[37m'

# Background color
BG_BLACK = '\033[40m'
BG_BLUE = '\033[44m'

# Reset code to return to default color
RESET = '\033[0m'

mapper = {
    BG_BLACK+RED: "#dd0000",
    BG_BLACK+GREEN: "#00dd00",
    BG_BLACK+YELLOW: "#dddd00",
    BG_BLACK+BLUE: "#0000ee",
    BG_BLACK+MAGENTA: "#aa00dd",
    BG_BLACK+CYAN: "#00dddd",
    BG_BLACK+WHITE: "#87CEEB",
    BG_BLUE+WHITE: "#ff7800"
}

# Reformat message
def reformat(message):
    for key, value in mapper.items():
        # Convert color to html snippet
        message = message.replace(key, f'<span style="color: {value}">')
    # For reset case    
    message = message.replace(RESET, '</span>')
    return message

In [9]:
# Preprocessor Agent: add a little preprocessing steps as layer before pass prompt to fine-tuned base model 

# Configurations
# The same base model used in fine-tuning
MODEL_NAME = "ollama/llama3.2"
BASE_URL = "http://localhost:11434"
# Prompt
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


class Preprocessor:
    def __init__(self):
        self.model_name = MODEL_NAME
        self.base_url = BASE_URL

    def messages_for(self, text: str) -> list[dict]:
        return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]

    def preprocess(self, text: str) -> str:
        # Validation
        if not text:
            return "No question found.", []
        
        # Handle dict case
        if isinstance(text, dict):
            question_text = text.get('summary', '')
            if not question_text:
                return "No question found.", []
        else:
            question_text = str(text).strip()
            if not question_text:
                return "No question found.", []
            
        messages = self.messages_for(question_text)
        response = completion(
            messages=messages,
            model=self.model_name,
            api_base=self.base_url,
        )
        # Return response
        return response.choices[0].message.content

In [10]:
# Test agent
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

### System:
Title: HyperX USB Condenser Mic
Category: Electronics
Brand: HyperX
Description: A high-quality, compact condenser microphone designed for clear and crisp audio.
Details: Features a USB-C connection for easy plug-and-play setup and reduced cable clutter.


In [11]:
# Work with fine-tuned model after deploy
# Test unit
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
Pricer = modal.Cls.from_name("best-deal-fine-tuned-service", "Pricer")
pricer = Pricer()
pricer.update_autoscaler(scaledown_window=1200)
reply = pricer.price.remote(text)
print(reply)

24.0


In [12]:
# Base agent: creating and record log messages
# Ensure info level logs are visible in notebooks and terminals
if not logging.getLogger().handlers:
    logging.basicConfig(level=logging.INFO, format="%(message)s")
else:
    logging.getLogger().setLevel(logging.INFO)

# Parent class for specialist agent which 
class Agent:
    # Foreground colors
    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    MAGENTA = '\033[35m'
    CYAN = '\033[36m'
    WHITE = '\033[37m'
    
    # Background color
    BG_BLACK = '\033[40m'
    
    # Reset color
    RESET = '\033[0m'

    # Attributes for the agent's name and color
    name: str = ""
    # The default color for the agent's log messages (white)
    # Later after inheritance gonna change with any color above
    color: str = '\033[37m'

    # Log messages with the agent's name and color
    def log(self, message):
        # Combine the background color and the agent's color for the log message
        color_code = self.BG_BLACK + self.color
        message = f"[{self.name}] {message}"
        # Access root of logger than pass the messeage 
        logging.getLogger().info(color_code + message + self.RESET)

In [13]:
# Specialist agent: runnin our fine-tuned model that's running remotely on Modal
class SpecialistAgent(Agent):
    import truststore
    truststore.inject_into_ssl()
    # define parent attribute inside child class
    name = "Specialist Agent"
    # Access red attribute
    color = Agent.RED

    def __init__(self):
        # Log massage
        self.log("Specialist Agent is initializing - connecting to modal")
        Pricer = modal.Cls.from_name("best-deal-fine-tuned-service", "Pricer")
        # Get instance of class
        self.pricer = Pricer()

    def price(self, description: str) -> float:
        if not description:
            return "No question found.", []
        
        # Handle dict case
        if isinstance(description, dict):
            question_text = description.get('summary', '')
            if not question_text:
                return "No question found.", []
        else:
            question_text = str(description).strip()
            if not question_text:
                return "No question found.", []
        # Log massage 
        self.log("Specialist Agent is calling remote fine-tuned model")
        result = self.pricer.price.remote(question_text)
        self.log(f"Specialist Agent completed - predicting ${result:.2f}")
        return result

In [14]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
agent = SpecialistAgent()
agent.price(text)

[Specialist Agent] Specialist Agent is initializing - connecting to modal
[Specialist Agent] Specialist Agent is calling remote fine-tuned model
[Specialist Agent] Specialist Agent completed - predicting $59.00


59.0

In [15]:
from answer import answer_question

RAG pipeline loaded.


In [16]:
# Test unit
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

### System:
Title: HyperX USB Condenser Microphone
Category: Electronics
Brand: HyperX
Description: High-quality condenser microphone providing clear and detailed audio for recording and streaming.
Details: Equipped with USB-C connectivity for easy plug-and-play setup.


In [17]:
answer_question(text)

(110.0,
 [Result(page_content='JLab Epic Talk USB Microphone | White | USB-C Output | Cardioid, Omni, Stereo, and Bi-Directional | 192k Sample Rate | Volume Control, Gain Control, and Quick Mute | 3.5mm AUX | Plug and Play\n\nTitle: JLab Epic Talk USB Microphone | White | USB-C Output | Cardioid, Omni, Stereo, and Bi-Directional | 192k Sample Rate | Volume Control, Gain Control, and Quick Mute | 3.5mm AUX | Plug and Play\n\nCategory: electronics\n\nDescription: Title: JLab Epic Talk Professional USB Microphone\nCategory: Electronics\nBrand: JLab\nDescription: High-resolution USB-C microphone designed for professional recording, streaming, and communication.\nDetails: Features 192kHz/24-bit sound quality, four selectable pickup patterns, adjustable gain and volume controls, plug-and-play setup, and versatile mounting options.', metadata={'source': 'product_1687', 'sku_codes': 'BI-DIRECTIONAL CARDIOID CONTROL MICROPHONE OUTPUT QUICK SAMPLE STEREO USB-C VOLUME WHITE', 'title': 'JLab Epic 

In [18]:
# Agent: answer phase of RAG
class FrontierAgent(Agent):
    # Configuraions
    # define parent attribute inside child class
    name = "Frontier Agent"
    # Access blue attribute
    color = Agent.BLUE
    def __init__(self):
        # Intialization
        self.log("Initializing Frontier Agent")

    def price(self,question: str,history: list[dict] | None = None) -> tuple[float | None, list]:

        description = question
        if not description:
            return "No question found.", []
                
        # Handle dict case
        if isinstance(description, dict):
            question_text = description.get('summary', '')
            if not question_text:
                return "No question found.", []
        else:
            question_text = str(description).strip()
            if not question_text:
                return "No question found.", []
        self.log("Frontier Agent is performing a RAG search of the Chroma datastore to find 10 similar products")
        estimate, chunks = answer_question(question_text)
        self.log(f"Frontier Agent completed - predicting ${estimate}")

        return estimate, chunks

In [19]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
agent = FrontierAgent()
agent.price(text)

[Frontier Agent] Initializing Frontier Agent
[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 10 similar products
[Frontier Agent] Frontier Agent completed - predicting $59.0


(59.0,
 [Result(page_content='Granvela USB Microphone for Windows PC/Laptop Plug and Play for Computer\n\nTitle: Granvela USB Microphone for Windows PC/Laptop Plug and Play for Computer\n\nCategory: electronics\n\nDescription: Title: Granvela USB Condenser Microphone for PC and Laptop  \nCategory: Electronics  \nBrand: Granvela  \nDescription: A plug-and-play USB microphone designed for clear audio recording and online communication.  \nDetails: Features omni-directional pickup, noise-canceling technology, compatible with Windows, MacOS, and other systems, with a 185cm USB cable and easy on/off control.', metadata={'source': 'product_4290', 'type': 'electronics', 'sku_codes': 'COMPUTER GRANVELA LAPTOP MICROPHONE WINDOWS', 'price': 17.99, 'product_id': 4290, 'title': 'Granvela USB Microphone for Windows PC/Laptop Plug and Play for Computer'}),
  Result(page_content='JLab Epic Talk USB Microphone | White | USB-C Output | Cardioid, Omni, Stereo, and Bi-Directional | 192k Sample Rate | Vol

In [20]:
# DNN  
class ResidualBlock(nn.Module): 
    def __init__(self, hidden_size, dropout_prob): 
        super(ResidualBlock, self).__init__() 
        self.block = nn.Sequential( 
            nn.Linear(hidden_size, hidden_size), 
            nn.LayerNorm(hidden_size), 
            nn.ReLU(), 
            nn.Dropout(dropout_prob), 
            nn.Linear(hidden_size, hidden_size), 
            nn.LayerNorm(hidden_size), 
        ) 
        self.relu = nn.ReLU() 
 
    def forward(self, x): 
        residual = x 
        out = self.block(x) 
        out += residual  # Skip connection 
        return self.relu(out) 
 
 
class DeepNeuralNetwork(nn.Module): 
    def __init__(self, input_size, num_layers=10, hidden_size=4096, dropout_prob=0.2): 
        super(DeepNeuralNetwork, self).__init__() 
 
        # First layer 
        self.input_layer = nn.Sequential( 
            nn.Linear(input_size, hidden_size), 
            nn.LayerNorm(hidden_size), 
            nn.ReLU(), 
            nn.Dropout(dropout_prob), 
        ) 
 
        # Residual blocks 
        self.residual_blocks = nn.ModuleList() 
        for i in range(num_layers - 2): 
            self.residual_blocks.append(ResidualBlock(hidden_size, dropout_prob)) 
 
        # Output layer 
        self.output_layer = nn.Linear(hidden_size, 1) 
 
    def forward(self, x): 
        x = self.input_layer(x) 
 
        for block in self.residual_blocks: 
            x = block(x) 
 
        return self.output_layer(x) 
 
 
Y_STD = 1.0328539609909058 
Y_MEAN = 4.434937953948975 
 
 
class DeepNeuralNetworkInference: 
    def __init__(self): 
        self.vectorizer = None 
        self.model = None 
        self.device = None 
 
        np.random.seed(42) 
        torch.manual_seed(42) 
        torch.cuda.manual_seed(42) 
 
    def setup(self): 
        self.vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True) 
        self.model = DeepNeuralNetwork(5000) 
        if torch.cuda.is_available(): 
            self.device = torch.device("cuda") 
        elif torch.backends.mps.is_available(): 
            self.device = torch.device("mps") 
        else: 
            self.device = torch.device("cpu") 
 
        logging.info(f"Neural Network is using {self.device}") 
 
        self.model.to(self.device) 
 
    def load(self, filename="deep_neural_network.pth"): 
        path = os.path.abspath(filename)
        self.model.load_state_dict(torch.load(path, map_location=self.device)) 
 
    def inference(self, text): 
        self.model.eval() 
        with torch.no_grad(): 
            vector = self.vectorizer.transform([text]) 
            vector = torch.FloatTensor(vector.toarray()).to(self.device) 
            pred = self.model(vector)[0] 
            result = torch.exp(pred * Y_STD + Y_MEAN) - 1 
            result = result.item() 
        return max(0, result)

In [21]:
# Test unit
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
# Create object
runner = DeepNeuralNetworkInference()
# Intializations
runner.setup()
# Load pytorch weights
runner.load("deep_neural_network.pth")

Neural Network is using cpu


In [22]:
runner.inference(text)

65.71004486083984

In [23]:
# Agent: DNN
class NeuralNetworkAgent(Agent):
    # Configuraions
    # define parent attribute inside child class
    name = "Neural Network Agent"
    color = Agent.MAGENTA

    def __init__(self):
        self.log("Neural Network Agent is initializing")
        # Create object
        self.neural_network = DeepNeuralNetworkInference()
        self.neural_network.setup()
        # Load weights
        self.neural_network.load("deep_neural_network.pth")
        self.log("Neural Network Agent is ready and weights are loaded")

    # Prediction
    def price(self, description: str) -> float:
        # Validation
        if not description:
            return "No question found."
        
        # Handle dict case
        if isinstance(description, dict):
            question_text = description.get('summary', '')
            if not question_text:
                return "No question found."
        else:
            question_text = str(description).strip()
            if not question_text:
                return "No question found."
        self.log("Neural Network Agent is starting a prediction")
        # Prediction 
        result = np.round(self.neural_network.inference(question_text), 2)
        self.log(f"Neural Network Agent completed - predicting ${result}")
        return result

In [24]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
agent = NeuralNetworkAgent()
agent.price(text)

[Neural Network Agent] Neural Network Agent is initializing
Neural Network is using cpu
[Neural Network Agent] Neural Network Agent is ready and weights are loaded
[Neural Network Agent] Neural Network Agent is starting a prediction
[Neural Network Agent] Neural Network Agent completed - predicting $106.09


106.09

In [25]:
frontier_agent = FrontierAgent()
specialist_agent = SpecialistAgent()
neural_network_agent = NeuralNetworkAgent()
preprocessor = Preprocessor()

[Frontier Agent] Initializing Frontier Agent
[Specialist Agent] Specialist Agent is initializing - connecting to modal
[Neural Network Agent] Neural Network Agent is initializing
Neural Network is using cpu
[Neural Network Agent] Neural Network Agent is ready and weights are loaded


In [26]:
# Work with ensemble concept
def ensemble(question):
    price1 = float(frontier_agent.price(question)[0])
    price2 = float(specialist_agent.price(question))
    price3 = float(neural_network_agent.price(question))
    # Ensemble formula
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [27]:
# Test unit
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)
ensemble(text)

[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 10 similar products


### System:
Title: HyperX Quadcast Gaming Microphone
Category: Electronics
Brand: HyperX
Description: A high-quality, plug-and-play USB condenser microphone for clear and immersive audio.
Details: Features a condenser capsule, 24-bit/48kHz audio, and a built-in pop filter to reduce plosives and background noise.


[Frontier Agent] Frontier Agent completed - predicting $130.0
[Specialist Agent] Specialist Agent is calling remote fine-tuned model
[Specialist Agent] Specialist Agent completed - predicting $140.00
[Neural Network Agent] Neural Network Agent is starting a prediction
[Neural Network Agent] Neural Network Agent completed - predicting $82.56


126.256

In [28]:
# Ensemble agent: apply ensembel technique with the three agents ---> RAG + LLAMA fine-tuned model + deep NN
# Import logging info
from concurrent.futures import ThreadPoolExecutor

# Preprocessor agent (agent to add a little preprocessing step) ---> muliptle models to estimate prices ---> specialist agent (estimate prices working directly with the fine-tuned base model), frontier agent (RAG with frontier model to estimate prices), neural network agent (estimate prices through deep NN)
class EnsembleAgent(Agent):
    # Configuraions
    # define parent attribute inside child class
    name = "Ensemble Agent"
    color = Agent.YELLOW

    def __init__(self):
        # Call objects of the four agents 
        self.log("Initializing Ensemble Agent")
        self.preprocessor = Preprocessor()
        self.specialist = SpecialistAgent()
        self.frontier = FrontierAgent()
        self.neural_network = NeuralNetworkAgent()
        self.log("Ensemble Agent is ready")

    def price(self, description: str) -> float:            
        import numpy as np
        self.log("Running Ensemble Agent - preprocessing text")
        question = self.preprocessor.preprocess(description)
        self.log(f"Pre-processed text using {self.preprocessor.model_name}")

        # These three calls are independent of each other, so run them concurrently
        # instead of waiting for each one to finish before starting the next
        with ThreadPoolExecutor(max_workers=3) as executor:
            frontier_future = executor.submit(lambda: float(self.frontier.price(question)[0]))
            specialist_future = executor.submit(lambda: float(self.specialist.price(question)))
            neural_network_future = executor.submit(lambda: float(self.neural_network.price(question)))

            frontier = frontier_future.result()
            specialist = specialist_future.result()
            neural_network = neural_network_future.result()
        # Ensemble linear regression combination formula
        combined = frontier * 0.8 + specialist * 0.1 + neural_network * 0.1
        self.log(f"Ensemble Agent complete - returning ${combined:.2f}")
        return np.round(combined, 2)

In [29]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
question = "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
agent = EnsembleAgent()
agent.price(question)

[Ensemble Agent] Initializing Ensemble Agent
[Specialist Agent] Specialist Agent is initializing - connecting to modal
[Frontier Agent] Initializing Frontier Agent
[Neural Network Agent] Neural Network Agent is initializing
Neural Network is using cpu
[Neural Network Agent] Neural Network Agent is ready and weights are loaded
[Ensemble Agent] Ensemble Agent is ready
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Ensemble Agent] Pre-processed text using ollama/llama3.2
[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 10 similar products
[Specialist Agent] Specialist Agent is calling remote fine-tuned model
[Neural Network Agent] Neural Network Agent is starting a prediction
[Neural Network Agent] Neural Network Agent completed - predicting $68.47
[Specialist Agent] Specialist Agent completed - predicting $24.00
[Frontier Agent] Frontier Agent completed - predicting $110.0
[Ensemble Agent] Ensemble Agent complete - returning $97.25

97.25

In [30]:
# Let's build scanner agent

In [31]:
# Deal scrapping script
# Links of RSS feeds to scrap the deals from dealnews.com
feeds = [
    "https://www.dealnews.com/c759/Health-Beauty/Beauty/?rss=1",
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c304/Home-Garden/Appliances/?rss=1",
    "https://www.dealnews.com/c196/Home-Garden/?rss=1",
    "https://www.dealnews.com/c859/Gift-Cards/?rss=1",
]




# Helper functions
# Scarpping snippet of html
# Summarize summary of the deal from html snippet
def extract(html_snippet: str) -> str:
    soup = BeautifulSoup(html_snippet, "html.parser")
    # Find this first div name
    snippet_div = soup.find("div", class_="snippet summary")
    if snippet_div:
        # Get its text
        description = snippet_div.get_text(strip=True)
        # Apply regex
        description = re.sub("<[^<]+?>", "", description)
        result = description.strip()
    else:
        # If nothing found return html snippet
        result = html_snippet
    return result.replace("\n", " ")



# Class to represent a raw data passed to LLM
# The input to LLM
# Phase of input of LLM: fetch 5 RSS feeds ---> feedparser ---> 10 entries per feed ---> ScrapedDeal objects so here we get the raw dataset to LLM
# ScrapedDeal preprocessing: title ---> extract text (description) from html of summary ---> request the link of page ---> scrape inside it and get content ---> get features and details
# This is how feed.entry looks like
"""
[
    {
        "title": "...",
        "summary": "...",
        "links": [...]
    }
]

"""
# Each RSS entry converted to this class 
class ScrapedDeal:
    # Attributes we need
    category: str
    title: str
    summary: str
    url: str
    details: str
    features: str

    def __init__(self, entry: Dict[str, str]):
        # Html entry
        # We need these attributes based on the output of feed parser
        # Parse entry we get
        self.title = entry["title"]
        # Html snippet
        self.summary = extract(entry["summary"])
        self.url = entry["links"][0]["href"]
        # Request the url then scrape the html to get the deatils and features
        stuff = requests.get(self.url).content
        soup = BeautifulSoup(stuff, "html.parser")
        # Find this first div name
        content = soup.find("div", class_="content-section").get_text()
        # Preprocess text
        content = content.replace("\nmore", "").replace("\n", " ")
        # If there is features word then make it as delimeter so features the same as deatils
        if "Features" in content:
            # Search for the first features word
            # Split details than features
            # Seperation happens because split produces list so two items with two elements of this list
            self.details, self.features = content.split("Features", 1)
        else:
            # If not found the word features then no features and the deatils is the content
            self.details = content
            self.features = ""
        # Trim the lenght    
        self.truncate()

    # Output format implementation
    def __repr__(self):
        return f"<{self.title}>"

    # Convert object to text
    # Helper function to convert ScrapedDeal object into text
    def describe(self):
        # Make title, deatils, features and url in description
        return f"Title: {self.title}\nDetails: {self.details.strip()}\nFeatures: {self.features.strip()}\nURL: {self.url}"
    
    # Trim title, details and summary
    # We need to limit lenght sent later to LLM
    def truncate(self):
        self.title = self.title[:100]
        self.details = self.details[:500]
        self.features = self.features[:500]
        
    # Output format implementation
    def __repr__(self):
        return f"<{self.title}>"

    # Convert object to text
    # Helper function to convert ScrapedDeal object into text
    def describe(self):
        # Make title, deatils, features and url in description
        return f"Title: {self.title}\nDetails: {self.details.strip()}\nFeatures: {self.features.strip()}\nURL: {self.url}"

    @classmethod
    def fetch(cls, show_progress: bool = False, max_workers: int = 10) -> List[Self]:
        # Retrieve deals and this is the actual work
        entries = []
        feed_iter = tqdm(feeds) if show_progress else feeds
        for feed_url in feed_iter:
            feed = feedparser.parse(feed_url)
            # Take first 10 deals of each RSS
            entries.extend(feed.entries[:10])

        # Run these concurrently instead of one at a time
        deals = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_entry = {executor.submit(cls, entry): entry for entry in entries}
            entry_iter = as_completed(future_to_entry)
            if show_progress:
                entry_iter = tqdm(entry_iter, total=len(future_to_entry))
            for future in entry_iter:
                try:
                    deals.append(future.result())
                except Exception as e:
                    # Don't let one bad page (404, timeout, malformed HTML) kill the whole run
                    logging.getLogger().warning(f"Skipping a deal entry due to error: {e}")
        return deals


# Output of LLM
class Deal(BaseModel):
    product_description: str = Field(
        description="Your clearly expressed summary of the product in 3-4 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a short paragraph of text for each item you choose."
    )
    price: float = Field(
        description="The actual price of this product, as advertised in the deal. Be sure to give the actual price; for example, if a deal is described as $100 off the usual $300 price, you should respond with $200"
    )
    url: str = Field(description="The URL of the deal, as provided in the input")

# The output of LLM 
# Sturctured output for list of deals from LLM
"""
{
  "deals": [
    {
      "product_description": "...",
      "price": 299.99,
      "url": "..."
    },
    {
      "product_description": "...",
      "price": 499.99,
      "url": "..."
    }
  ]
}
"""


# Phase of output of LLM: DealSelection object
# DealSelection object: list of Deal objects
class DealSelection(BaseModel):
    # List of all deals
    deals: List[Deal] = Field(
        description="Your selection of the 5 deals that have the most detailed, high quality description and the most clear price. You should be confident that the price reflects the deal, that it is a good deal, with a clear description"
    )

# The strctured output of LLM after converting deal to opportunity
"""
{
  "deal": {
    "product_description": "MacBook Air M4...",
    "price": 799.99,
    "url": "https://..."
  },
  "estimate": 1000.0,
  "discount": 20.001
}

"""
class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float

In [34]:
# Scrap deals
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 50/50 [00:21<00:00,  2.35it/s]


In [35]:
# Invitigate deals
len(deals)

50

In [36]:
# Output of ScrapedDeal object ---> title
deals[10]

<Ortizan C7 Powered Studio Monitor Speakers Pair for $73 + free shipping>

In [37]:
print(deals[20].describe())

Title: Insignia 165-Can Built-In Beverage Cooler for $400 + free shipping
Details: As one of its daily deals, Best Buy has this Insignia beverage cooler for $399.99, down from $649.99. It holds up to 165 12-oz. cans across three adjustable wire shelves and includes automatic defrosting and a reversible door for flexible installation.  Buy Now at Best Buy
Features: 
URL: https://www.dealnews.com/Insignia-165-Can-Built-In-Beverage-Cooler-for-400-free-shipping/22209615.html?iref=rss-c304


In [38]:
# Scan the top n deals with LLM
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [39]:
# Create user prompt
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    # Deatils of products
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [40]:
user_prompt = make_user_prompt(deals)
# truncate to first 2000 char
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Personal Care Deals at Woot: Up to 68% off + free shipping w/ Prime
Details: Woot's Personal Care sale covers a wide range of everyday essentials from brands like Gillette, Crest, Secret, and Aveeno, with discounts reaching up to 68% off. Refurbished Theragun massage guns and a TheraFace Pro are also part of the mix, starting at $89.99. Most items are priced under $15, making t

In [41]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

In [42]:
# Test unit
GENERATOR_MODEL = "gpt-5-mini"
#Don't forget .parse so we can pass structured output back which is DealSelection object from deals agent
response = openai.chat.completions.parse(model=GENERATOR_MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='The Samsung Galaxy S25 Edge unlocked smartphone with 256GB storage features a 6.7-inch display, a titanium chassis protected by Corning Gorilla Glass Ceramic 2, and a 200MP main camera. It’s factory-unlocked for use on any carrier and includes high-end materials and imaging hardware suited for power users who want flagship performance and premium build quality.', price=549.99, url='https://www.dealnews.com/Unlocked-Samsung-Galaxy-S25-Edge-256-GB-Phone-for-550-512-GB-for-650-free-shipping-w-Prime/22209630.html?iref=rss-c142'), Deal(product_description='The Dreame Aqua 10 Ultra is a high-end robot vacuum and mop that uses a self-emptying, self-cleaning PowerDock capable of washing its mop roller at 212°F and providing up to 100 days between manual dust bag changes. The robot delivers strong 30,000Pa suction, a detangling roller brush to reduce hair clogs, and water-based cleaning with obstacle clearance via FlexRise suspension for multi-surf

In [43]:
# Invistigate output of LLM
for deal in results.deals:
    # The structured ouput we expected to see for the selected outputs
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The Samsung Galaxy S25 Edge unlocked smartphone with 256GB storage features a 6.7-inch display, a titanium chassis protected by Corning Gorilla Glass Ceramic 2, and a 200MP main camera. It’s factory-unlocked for use on any carrier and includes high-end materials and imaging hardware suited for power users who want flagship performance and premium build quality.
549.99
https://www.dealnews.com/Unlocked-Samsung-Galaxy-S25-Edge-256-GB-Phone-for-550-512-GB-for-650-free-shipping-w-Prime/22209630.html?iref=rss-c142

The Dreame Aqua 10 Ultra is a high-end robot vacuum and mop that uses a self-emptying, self-cleaning PowerDock capable of washing its mop roller at 212°F and providing up to 100 days between manual dust bag changes. The robot delivers strong 30,000Pa suction, a detangling roller brush to reduce hair clogs, and water-based cleaning with obstacle clearance via FlexRise suspension for multi-surface cleaning.
650.0
https://www.dealnews.com/Dreame-Aqua-10-Ultra-Robot-Vacuum-and-Mop-fo

In [44]:
# Scanner agent: scrape deal and call LLM to select the top 5 of them returning them in structured output
class ScannerAgent(Agent):
    # Configurations
    GENERATOR_MODEL = "gpt-5-mini"
    SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
    Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
    Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
    Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
    """
    USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
    You should rephrase the description to be a summary of the product itself, not the terms of the deal.
    Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
    Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

    Deals:

    """
    USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."
    name = "Scanner Agent"
    color = Agent.CYAN

    def __init__(self):
        self.log("Scanner Agent is initializing")
        self.openai = OpenAI()
        self.log("Scanner Agent is ready")

    def fetch_deals(self, memory) -> List[ScrapedDeal]:
        self.log("Scanner Agent is about to fetch deals from RSS feed")
        urls = [opp.deal.url for opp in memory]
        scraped = ScrapedDeal.fetch()
        # Work with new results only
        result = [scrape for scrape in scraped if scrape.url not in urls]
        self.log(f"Scanner Agent received {len(result)} deals not already scraped")
        return result

    def make_user_prompt(self, scraped) -> str:
        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
        user_prompt += self.USER_PROMPT_SUFFIX
        return user_prompt       
    
    def scan(self, memory: List[str] = []) -> Optional[DealSelection]:
        scraped = self.fetch_deals(memory)
        if scraped:
            user_prompt = self.make_user_prompt(scraped)
            self.log("Scanner Agent is calling OpenAI using Structured Outputs")
            result = self.openai.chat.completions.parse(
                model=self.GENERATOR_MODEL,
                messages=[
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=DealSelection,
                reasoning_effort="minimal",
            )
            result = result.choices[0].message.parsed
            result.deals = [deal for deal in result.deals if deal.price > 0]
            self.log(
                f"Scanner Agent received {len(result.deals)} selected deals with price>0 from OpenAI"
            )
            for deal in result.deals:
                print(deal.product_description)
                print(deal.price)
                print(deal.url)
                print()
            return result     
        return None

    # Testing fake data 
    # Return JSON into Deals objects (DealSelection)
    def test_scan(self, memory: List[str] = []) -> Optional[DealSelection]:
        results = {
            "deals": [
                {
                    "product_description": "The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.",
                    "price": 178,
                    "url": "https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142",
                },
                {
                    "product_description": "The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.",
                    "price": 30,
                    "url": "https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39",
                },
                {
                    "product_description": "The Lenovo IdeaPad Slim 5 laptop is powered by a 7th generation AMD Ryzen 5 8645HS 6-core CPU, offering efficient performance for multitasking and demanding applications. It features a 16-inch touch display with a resolution of 1920x1080, ensuring bright and vivid visuals. Accompanied by 16GB of RAM and a 512GB SSD, the laptop provides ample speed and storage for all your files. This model is designed to handle everyday tasks with ease while delivering an enjoyable user experience.",
                    "price": 446,
                    "url": "https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-5-7-th-Gen-Ryzen-5-16-Touch-Laptop/485068.html?iref=rss-c39",
                },
                {
                    "product_description": "The Dell G15 gaming laptop is equipped with a 6th-generation AMD Ryzen 5 7640HS 6-Core CPU, providing powerful performance for gaming and content creation. It features a 15.6-inch 1080p display with a 120Hz refresh rate, allowing for smooth and responsive gameplay. With 16GB of RAM and a substantial 1TB NVMe M.2 SSD, this laptop ensures speedy performance and plenty of storage for games and applications. Additionally, it includes the Nvidia GeForce RTX 3050 GPU for enhanced graphics and gaming experiences.",
                    "price": 650,
                    "url": "https://www.dealnews.com/products/Dell/Dell-G15-Ryzen-5-15-6-Gaming-Laptop-w-Nvidia-RTX-3050/485067.html?iref=rss-c39",
                },
            ]
        }
        # Use key args to pass unknown number of arguments
        return DealSelection(**results)


In [45]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
agent = ScannerAgent()
result = agent.test_scan()
print(result)

[Scanner Agent] Scanner Agent is initializing
[Scanner Agent] Scanner Agent is ready


deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control,

In [46]:
for deal in result.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.
178.0
https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142

The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed

In [47]:
# Let's build messageing agent
# Connect with Pushover
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [48]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with g
Pushover token found and starts with a


In [49]:
def push(message):
    print(f"Push: {message}")
    # Create payload which has the authorization and the messgage itself
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    # Post the message so pushin it through API url
    requests.post(pushover_url, data=payload)

In [50]:
# Test unit
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [51]:
# Messaging agent: pushes notification to your phone when got deals
# We have two kinds of notifications 
# First one from opportunity and second one from agents 
class MessagingAgent(Agent):
    # define parent attribute inside child class
    name = "Messaging Agent"
    color = Agent.WHITE
    # Configurations
    GENERATOR_MODEL = "gpt-5-mini"
    pushover_url = "https://api.pushover.net/1/messages.json"

    def __init__(self):
        # Connect with Pushover
        self.log("Messaging Agent is initializing")
        self.pushover_user = os.getenv("PUSHOVER_USER", "your-pushover-user-if-not-using-env")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "your-pushover-user-if-not-using-env")
        self.log("Messaging Agent has initialized Pushover and OpenAI")

    # helper function
    # Push messages
    def push(self, text):
        self.log("Messaging Agent is sending a push notification")
        # Create payload which has the authorization and the messgage itself
        payload = {"user": self.pushover_user, "token": self.pushover_token, "message": text, "sound": "cashregister", }
        # Post the message so pushin it through API url
        requests.post(self.pushover_url, data=payload)

    # Alrert with opportunity
    def alert(self, opportunity: Opportunity):
        # Format message
        text = f"Deal Alert! Price=${opportunity.deal.price:.2f}, "
        text += f"Estimate=${opportunity.estimate:.2f}, "
        text += f"Discount=${opportunity.discount:.2f} :"
        # Trnucate description
        text += opportunity.deal.product_description[:10] + "... "
        # The most important part the URL itself of the opportunity
        text += opportunity.deal.url
        # Push
        self.push(text)
        self.log("Messaging Agent has completed")

    # Use LLM to summarize the agent messages to user
    def craft_message(
        self, description: str, deal_price: float, estimated_true_value: float
    ) -> str:
        user_prompt = "Please summarize this great deal in 2-3 sentences to be sent as an exciting push notification alerting the user about this deal.\n"
        user_prompt += f"Item Description: {description}\nOffered Price: {deal_price}\nEstimated true value: {estimated_true_value}"
        user_prompt += "\n\nRespond only with the 2-3 sentence message which will be used to alert & excite the user about this deal"
        response = completion(
            model=self.GENERATOR_MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content

    # Use the LLM to notify
    def notify(self, description: str, deal_price: float, estimated_true_value: float, url: str):
        self.log("Messaging Agent is using OpenAI to craft the message")
        text = self.craft_message(description, deal_price, estimated_true_value)
        # Formulate notification
        self.push(text[:200] + "... " + url)
        self.log("Messaging Agent has completed")


In [52]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
agent = MessagingAgent()
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

[Messaging Agent] Messaging Agent is initializing
[Messaging Agent] Messaging Agent has initialized Pushover and OpenAI
[Messaging Agent] Messaging Agent is using OpenAI to craft the message
[Messaging Agent] Messaging Agent is sending a push notification
[Messaging Agent] Messaging Agent has completed


In [53]:
# Let's build planner agent
# Build functions
def scan_the_internet_for_bargains() -> str:
    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    # Convert to JSON 
    return result.model_dump_json()

In [54]:
scan_the_internet_for_bargains()

Fake function to scan the internet - this returns a hardcoded set of deals


'{"deals":[{"product_description":"The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku\'s operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.","price":178.0,"url":"https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},{"product_description":"The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom contro

In [55]:
def estimate_true_value(description: str) -> str:
    print(f"Fake function to estimating true value of {description[:20]}... - this always returns $300")
    return f"Product {description} has an estimated true value of $300"

In [56]:
def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    print(f"Fake function to notify user of {description} which costs {deal_price} and estimate is {estimated_true_value}")
    return "notification sent ok"

In [57]:
notify_user_of_deal("a new iphone", 100, 1000, "https://www.apple.com/iphone")

Fake function to notify user of a new iphone which costs 100 and estimate is 1000


'notification sent ok'

In [61]:
# Tool schema 
scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [59]:
# List of tool schemas 
tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
 ]

In [62]:
tools

[{'type': 'function',
  'function': {'name': 'scan_the_internet_for_bargains',
   'description': 'Returns top bargains scraped from the internet along with the price each item is being offered for',
   'parameters': {'type': 'object',
    'properties': {},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'estimate_true_value',
   'description': 'Given the description of an item, estimate how much it is actually worth',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description': 'The description of the item to be estimated'}},
    'required': ['description'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'notify_user_of_deal',
   'description': 'Send the user a push notification about the single most compelling deal; only call this one time',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string',
      'description

In [ ]:
# Handle tool call ---> call relevant functions
# message: LLM response
def handle_tool_call(message):
    # Case of tools
    responses = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        if tool:
            result = tool(**arguments)
        else:
            result = {"error": f"Unknown tool: {tool_name}"}
        # Return the tool result back to the LLM
        responses.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return responses

In [65]:
# Send the list of tools to LLM ---> structured tool call ---> handle tool call ---> call relevant functions
system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

In [66]:
messages

[{'role': 'system',
  'content': 'You find great deals on bargain products using your tools, and notify the user of the best bargain.'},
 {'role': 'user',
  'content': '\nFirst, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.\nThen pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.\nThen just reply OK to indicate success.\n'}]

In [67]:
GENERATOR_MODEL = "gpt-5-mini"
done = False
# Our agent loop later
while not done:
    response = openai.chat.completions.create(model=GENERATOR_MODEL, messages=messages)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # Call the tool getting response to be handled
        response = handle_tool_call(message)
        # Build conversation history
        # Add assitant message to history
        messages.append(message)
        # Add tool result to history
        # Tool handling returns list of responses
        messages.extend(response)
    else:
        done = True
response.choices[0].message.content

'I can’t browse the live web from here, so I can’t scan the internet or use external tools to find and value current bargain deals. I can, however, do one of the following for you — tell me which you prefer:\n\n1) You paste product links or names (and the listed sale prices) and I’ll estimate each item’s “true value” using recent market comparables, MSRP, historical sale prices, used-market values, warranty/return factors, and total cost of ownership, then pick the best bargain.\n\n2) Tell me the product categories, region/currency, and any price range or retailers to target (e.g., “laptops, US, <$500, Amazon/BestBuy”). I’ll give step-by-step search instructions plus the valuation criteria and how to pick the single best bargain, or I can simulate likely bargains using typical market data.\n\n3) If you can enable a browsing tool or give API access, I can use that to perform a live scan and valuation.\n\nIf you want option 1, paste up to ~10 links or product titles/prices and I’ll do th

In [70]:
# Autonomous planning agent: orchesrate other agents getting the raw data of scanner as Deal object and convert them into Opportunity object then pushing it as notifcation
# planning agent ---> scanner tool ---> ensemble tool ---> messaging tool
# Here discount ---> return opportunities ---> sort opportunities ---> get the best ---> compare threshold ---> alert notification is done through messaging tool
class AutonomousPlanningAgent(Agent):
    # define parent attribute inside child class
    name = "Autonomous Planning Agent"
    color = Agent.GREEN
    # Configurations
    GENERATOR_MODEL = "gpt-5.1"

    def __init__(self):
        # Intialization
        self.log("Autonomous Planning Agent is initializing")
        # Scanner agent
        self.scanner = ScannerAgent()
        # Ensemble agent
        self.ensemble = EnsembleAgent()
        # Messaging agent
        self.messenger = MessagingAgent()
        # For tool calling
        self.openai = OpenAI()
        # For mempry
        self.memory = None
        self.opportunity = None
        self.log("Autonomous Planning Agent is ready")

    # Build functions 
    # We have three agents as three tools 
    # Get deals scrapped also taking in consideration the seen deals
    # Scanner agent
    def scan_the_internet_for_bargains(self) -> str:
        """
        Run the tool to scan
        """
        self.log("Autonomous Planning agent is calling scanner")
        results = self.scanner.scan(memory=self.memory)
        # Convert JSON into text
        return results.model_dump_json() if results else "No deals found"

    # Here we pass deal from scanner agent and return esrtimated price
    def estimate_true_value(self, description: str) -> str:
        """
        Run the tool to estimate true value
        """
        self.log("Autonomous Planning agent is estimating value via Ensemble Agent")
        # First pass deal and return predicted price based on deal description
        estimate = self.ensemble.price(description)
        return f"The estimated true value of {description} is {estimate}"

    # When we call this function this means we already have the best deal 
    def notify_user_of_deal(
        self, description: str, deal_price: float, estimated_true_value: float, url: str
    ) -> Dict:
        if self.opportunity:
            # If we already have opportunities and we go here then it's second time of pushing notifications
            self.log("Autonomous Planning agent is trying to notify the user a 2nd time; ignoring")
        else:
            # First time to push notification
            self.log("Autonomous Planning agent is notifying user")
            self.messenger.notify(description, deal_price, estimated_true_value, url)
            # Create a new Deal object
            deal = Deal(product_description=description, price=deal_price, url=url)
            discount = estimated_true_value - deal_price
            self.opportunity = Opportunity(
                deal=deal, estimate=estimated_true_value, discount=discount
            )
        return "Notification sent ok"

    # Tool schema 
    scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    }

    estimate_function = {
        "name": "estimate_true_value",
        "description": "Given the description of an item, estimate how much it is actually worth",
        "parameters": {
            "type": "object",
            "properties": {
                "description": {
                    "type": "string",
                    "description": "The description of the item to be estimated",
                },
            },
            "required": ["description"],
            "additionalProperties": False,
        },
    }

    notify_function = {
        "name": "notify_user_of_deal",
        "description": "Send the user a push notification about the single most compelling deal; only call this one time",
        "parameters": {
            "type": "object",
            "properties": {
                "description": {
                    "type": "string",
                    "description": "The description of the item itself scraped from the internet",
                },
                "deal_price": {
                    "type": "number",
                    "description": "The price offered by this deal scraped from the internet",
                },
                "estimated_true_value": {
                    "type": "number",
                    "description": "The estimated actual value that this is worth",
                },
                "url": {
                    "type": "string",
                    "description": "The URL of this deal as scraped from the internet",
                },
            },
            "required": ["description", "deal_price", "estimated_true_value", "url"],
            "additionalProperties": False,
        },
    }

    # Create list of tool schemas
    def get_tools(self):
        return [
            {"type": "function", "function": self.scan_function},
            {"type": "function", "function": self.estimate_function},
            {"type": "function", "function": self.notify_function},
        ]

    # Handle tool call ---> call relevant functions
    # message: LLM response
    def handle_tool_call(self, message):
        mapping = {
            "scan_the_internet_for_bargains": self.scan_the_internet_for_bargains,
            "estimate_true_value": self.estimate_true_value,
            "notify_user_of_deal": self.notify_user_of_deal,
        }

        def run_one(tool_call):
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool = mapping.get(tool_name)
            if tool:
                result = tool(**arguments)
            else:
                result = {"error": f"Unknown tool: {tool_name}"}
            return {
                    "role": "tool",
                    # Here the resturned value is text
                    "content": result,
                    "tool_call_id": tool_call.id
                    }

        # Here we work with threading so in case of only one tool call we just call the function with no need of threading
        tool_calls = list(message.tool_calls)
        if len(tool_calls) == 1:
            results = [run_one(tool_calls[0])]
        else:
            # Loop over all calls
            with ThreadPoolExecutor(max_workers=len(tool_calls)) as executor:
                # Return the tool result back to the LLM
                results = list(executor.map(run_one, tool_calls))
        return results

    # Send the list of tools to LLM ---> structured tool call ---> handle tool call ---> call relevant functions
    # Here we depend on pormpt instead of hard coded threshold so LLMs get the best deal
    system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
    user_message = """
    First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
    Then pick the single most compelling deal where the estimated true value exceeds the deal price by more than $50, and use your tool to notify the user.
    If no deal meets this threshold, do not call the notification tool, and just reply OK.
    Then just reply OK to indicate success.
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    # LLM decide the flow and which agent to be called
    def plan(self, memory: List[str] = []) -> Optional[Opportunity]:
        self.log("Autonomous Planning Agent is kicking off a run")
        # Intialize memory
        self.memory = memory
        # Intialize opportunitues
        self.opportunity = None
        # Intialize messages
        messages = self.messages[:]
        # Create loops to get all responses with all tool calling 
        done = False
        # Our agent loop
        while not done:
            response = self.openai.chat.completions.create(
                model=self.GENERATOR_MODEL, messages=messages, tools=self.get_tools()
            )
            if response.choices[0].finish_reason == "tool_calls":
                message = response.choices[0].message
                results = self.handle_tool_call(message)
                messages.append(message)
                messages.extend(results)
            else:
                # To break loop if no tool 
                done = True
        reply = response.choices[0].message.content
        self.log(f"Autonomous Planning Agent completed with: {reply}")
        return self.opportunity

In [71]:
# Test agent
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
agent = AutonomousPlanningAgent()
result = agent.plan()

[Autonomous Planning Agent] Autonomous Planning Agent is initializing
[Scanner Agent] Scanner Agent is initializing
[Scanner Agent] Scanner Agent is ready
[Ensemble Agent] Initializing Ensemble Agent
[Specialist Agent] Specialist Agent is initializing - connecting to modal
[Frontier Agent] Initializing Frontier Agent
[Neural Network Agent] Neural Network Agent is initializing
Neural Network is using cpu
[Neural Network Agent] Neural Network Agent is ready and weights are loaded
[Ensemble Agent] Ensemble Agent is ready
[Messaging Agent] Messaging Agent is initializing
[Messaging Agent] Messaging Agent has initialized Pushover and OpenAI
[Autonomous Planning Agent] Autonomous Planning Agent is ready
[Autonomous Planning Agent] Autonomous Planning Agent is kicking off a run
[Autonomous Planning Agent] Autonomous Planning agent is calling scanner
[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
[Scanner Agent] Scanner Agent received 50 deals not already scraped
[Scanner 

Incamity Electric Foot Callus Remover is a rechargeable handheld device designed to smooth and remove hardened skin. It includes a 2000mAh battery with an LED charge display, USB-C fast charging, and three adjustable speed settings to tailor exfoliation. The kit ships with coarse and fine replaceable sandpaper rollers and includes a foot care cream stick for aftercare, providing up to several hours of runtime per charge.
9.99
https://www.dealnews.com/Incamity-Electric-Foot-Callus-Remover-for-10-free-shipping-w-Prime/22192141.html?iref=rss-c759

Dreame Aqua 10 Ultra Robot Vacuum and Mop is a high-end autonomous floor cleaner with a self-emptying, self-cleaning dock that washes the mop roller at high temperature. It offers 30,000Pa suction, a detangling roller brush to prevent hair clogs, FlexRise air suspension to handle obstacles up to about 3.15 inches, and long maintenance intervals—up to 100 days between dust-bag changes—making it suitable for large or pet-owner households.
650.0
ht

[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
[Autonomous Planning Agent] Autonomous Planning agent is estimating value via Ensemble Agent
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Ensemble Agent] Running Ensemble Agent - preprocessing text
[Ensemble Agent] Pre-processed text using ollama/llama3.2
[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 10 similar products
[Specialist Agent] Specialist Agent is calling remote fine-tu

In [73]:
print(result.estimate)

810.06


In [74]:
# Let's build deal agent framework

In [75]:
# Let's build demo UI for the whole system
# This is our agent framework
# Framework orchestrates system lifecycle
# Memory managed by deal aganet framework and loop managed by autonomous planning agent
# This isn't single agent as agent responsible of specific task but it orchesrtates the autonomous planning agent and manages the agent memory
'''
DealAgentFramework
│
├── Memory Management
│      ├── Read memory
│      ├── Write memory
│      └── Reset memory
│
└── Autonomous Planning Agent
       │
       ├── Scanner Agent
       ├── Ensemble Agent
       └── Messaging Agent
'''
'''
              Deal agent framework
                         │
         ┌───────────────┴────────────────┐
         │                                │
         ▼                                ▼
   Initialize logging               Read memory
         │                                │
         └───────────────┬────────────────┘
                         │
                         ▼
          Autonomous planning agent
                         │
                  GPT tool loop
                         │
        ┌────────────────┼────────────────┐
        ▼                ▼                ▼
 Scanner tool     Ensemble tool    Messaging tool
        │                │                │
        └────────────────┴────────────────┘
                         │
                    Opportunity
                         │
                         ▼
                Deal agent framework
                         │
                    Write memory
                         │
                         ▼
                    End program
'''                   



# Configurations
load_dotenv(override=True)

# Colors for logging
BG_BLUE = "\033[44m"
WHITE = "\033[37m"
RESET = "\033[0m"

# Initialize logging helper function
# Here we use different log function rather than agent.py as it's more advanced log with time stamp, level and messages
# The framework is the entry point of the program so we need professional logging
def init_logging():
    # Get root of logger
    root = logging.getLogger()
    # Any king of message ---> info, warning or error appear
    root.setLevel(logging.INFO)
    # Make messages appear in terminal
    handler = logging.StreamHandler(sys.stdout)
    # Any king of message ---> info, warning or error appear
    handler.setLevel(logging.INFO)
    # Format of message
    formatter = logging.Formatter(
        "[%(asctime)s] [Agents] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S %z",
    )
    handler.setFormatter(formatter)
    root.addHandler(handler)

class DealAgentFramework:
    # Configurations
    # Memory
    MEMORY_FILENAME = "memory.json"

    def __init__(self):
        # Intializations
        # Initialize logging 
        init_logging()
        # Read memory from file
        self.memory = self.read_memory()
        self.planner = None

    # Call the Autonomous planning agent
    def init_agents_as_needed(self):
        # If we don't call it yet
        if not self.planner:
            self.log("Initializing Agent Framework")
            self.planner = AutonomousPlanningAgent()
            self.log("Agent Framework is ready")

    # Read memory helper function
    def read_memory(self) -> List[Opportunity]:
        # Make sure we have file to read memeory from it unless we retrun empty list
        if os.path.exists(self.MEMORY_FILENAME):
            with open(self.MEMORY_FILENAME, "r") as file:
                # Read JSON
                data = json.load(file)
            # JSON ---> dict of items ---> Opportunity    
            # Memory stores only previously processed (seen) opportunities
            # ScannerAgent which orchestrated by autonomous planning agent expects memory to contain Opportunity objects 
            opportunities = [Opportunity(**item) for item in data]
            return opportunities
        return []

    # Write memory
    def write_memory(self) -> None:
        # Be careful first thing every happen is write then in next requests read will take part in
        # Opportunity ---> dict of items ---> JSON
        data = [opportunity.model_dump() for opportunity in self.memory]
        with open(self.MEMORY_FILENAME, "w") as file:
            # Write JSON after first two elements
            # We keep the two default elemenets even in reset as bussiness decision
            json.dump(data, file, indent=2)

    # Reset Memory
    @classmethod
    def reset_memory(cls) -> None:
        # We have to read ---> truncate ---> write
        data = []
        # Make sure we have file to read
        if os.path.exists(cls.MEMORY_FILENAME):
            with open(cls.MEMORY_FILENAME, "r") as file:
                # Read JSON
                data = json.load(file)
        # Keep first two items then write them        
        truncated = data[:2]
        with open(cls.MEMORY_FILENAME, "w") as file:
            json.dump(truncated, file, indent=2)

    # It's the same like log messages with the agent's name and color in agent.py
    def log(self, message: str):
        text = BG_BLUE + WHITE + "[Agent Framework] " + message + RESET
        logging.info(text)
   
    # Main function to run the framework               
    def run(self) -> List[Opportunity]:
        # Create instance of autonomous planning agent
        self.init_agents_as_needed()
        logging.info("Kicking off Planning Agent")
        result = self.planner.plan(memory=self.memory)
        logging.info(f"Planning Agent has completed and returned: {result}")
        if result:
            # Seen data to be stored in memory
            self.memory.append(result)
            self.write_memory()
        return self.memory

In [76]:
# Test framework
litellm.suppress_debug_info = True
for name in ["openai", "openai._base_client", "httpx", "httpcore", "LiteLLM"]:
    logging.getLogger(name).setLevel(logging.CRITICAL)
agent = DealAgentFramework().run()

[Agent Framework] Initializing Agent Framework


[2026-09-23 14:52:15 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework


[Autonomous Planning Agent] Autonomous Planning Agent is initializing


[2026-09-23 14:52:15 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing


[Scanner Agent] Scanner Agent is initializing


...

[2026-09-23 14:55:30 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed


[Autonomous Planning Agent] Autonomous Planning Agent completed with: OK


[2026-09-23 14:55:32 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent completed with: OK


Planning Agent has completed and returned: deal=Deal(product_description='Pair of Ortizan C7 powered studio monitor speakers designed for nearfield mixing and desktop listening. Each speaker combines a 3.5" carbon-fiber mid-bass driver with a 0.75" silk-dome tweeter, a built-in 24-bit DAC for USB audio, Bluetooth 5.3 wireless streaming, and multiple wired inputs including RCA, 3.5mm AUX, and a 6.35mm TRS balanced input for connecting mixers or instruments. The monitors include front-panel headphone output and controls for convenient desktop use.', price=73.0, url='https://www.dealnews.com/Ortizan-C7-Powered-Studio-Monitor-Speakers-Pair-for-73-free-shipping/22209663.html?iref=rss-c142') estimate=301.28 discount=228.27999999999997


[2026-09-23 14:55:32 +0300] [Agents] [INFO] Planning Agent has completed and returned: deal=Deal(product_description='Pair of Ortizan C7 powered studio monitor speakers designed for nearfield mixing and desktop listening. Each speaker combines a 3.5" carbon-fiber mid-bass driver with a 0.75" silk-dome tweeter, a built-in 24-bit DAC for USB audio, Bluetooth 5.3 wireless streaming, and multiple wired inputs including RCA, 3.5mm AUX, and a 6.35mm TRS balanced input for connecting mixers or instruments. The monitors include front-panel headphone output and controls for convenient desktop use.', price=73.0, url='https://www.dealnews.com/Ortizan-C7-Powered-Studio-Monitor-Speakers-Pair-for-73-free-shipping/22209663.html?iref=rss-c142') estimate=301.28 discount=228.27999999999997


In [77]:
# Read memory after getting opportunity and writing it to memory file
memory = DealAgentFramework().read_memory()

In [78]:
memory

[Opportunity(deal=Deal(product_description='The Samsung Galaxy Watch Ultra is a premium 47mm LTE Titanium smartwatch designed for both style and functionality. It features a circular display made with durable materials suitable for outdoor activities, providing GPS tracking, health monitoring, and custom apps for various needs. The robust design integrates a range of smart features including notifications, music control, and heart rate tracking, making it an ideal companion for fitness enthusiasts and tech-savvy users alike.', price=350.0, url='https://www.dealnews.com/Samsung-Galaxy-Watch-Ultra-47-mm-LTE-Titanium-Smartwatch-up-to-350-off-w-Trade-in-free-shipping/21663266.html?iref=rss-c142'), estimate=773.8138460593241, discount=423.8138460593241),
 Opportunity(deal=Deal(product_description="The Refurbished Unlocked Apple iPhone 14 Pro Max offers an impressive 256GB storage and a huge display, perfect for both media consumption and productivity. Enjoy advanced camera technology for st

In [79]:
DealAgentFramework().reset_memory()

In [80]:
# Read memory after reset
memory = DealAgentFramework().read_memory()

In [81]:
memory

[Opportunity(deal=Deal(product_description='The Samsung Galaxy Watch Ultra is a premium 47mm LTE Titanium smartwatch designed for both style and functionality. It features a circular display made with durable materials suitable for outdoor activities, providing GPS tracking, health monitoring, and custom apps for various needs. The robust design integrates a range of smart features including notifications, music control, and heart rate tracking, making it an ideal companion for fitness enthusiasts and tech-savvy users alike.', price=350.0, url='https://www.dealnews.com/Samsung-Galaxy-Watch-Ultra-47-mm-LTE-Titanium-Smartwatch-up-to-350-off-w-Trade-in-free-shipping/21663266.html?iref=rss-c142'), estimate=773.8138460593241, discount=423.8138460593241),
 Opportunity(deal=Deal(product_description="The Refurbished Unlocked Apple iPhone 14 Pro Max offers an impressive 256GB storage and a huge display, perfect for both media consumption and productivity. Enjoy advanced camera technology for st

In [82]:
# Let's build UI

In [ ]:
# Demo of final script
agent_framework = DealAgentFramework()
# Skip memory read and write for demo purposes
agent_framework.init_agents_as_needed()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [[opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url] for opp in opps]

    def do_select(opportunities, selected_index: gr.SelectData):
            row = selected_index.index[0]
            opportunity = opportunities[row]
            agent_framework.planner.messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            row_count=(1, "dynamic"),
            # make sure it's column_count not col_count
            column_count=(5, "fixed"),   
            max_height=400,
            
        )
    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

[Agent Framework] Initializing Agent Framework


[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Agent Framework] Initializing Agent Framework


[Autonomous Planning Agent] Autonomous Planning Agent is initializing


[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing
[2026-09-23 14:56:24 +0300] [Agents] [INFO] [Autonomous Planning Agent] Autonomous Planning Agent is initializing


[Scanner Agent] Scanner Agent is initializing


...

[2026-09-23 14:56:54 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:54 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:54 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:54 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:54 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed


[Messaging Agent] Messaging Agent is sending a push notification


[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent is sending a push notification


[Messaging Agent] Messaging Agent has completed


[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed
[2026-09-23 14:56:55 +0300] [Agents] [INFO] [Messaging Agent] Messaging Agent has completed


In [ ]:
App().run()